# Expanding the Urdu OCR Dataset with UTRSet

This notebook adds real and synthetic Urdu text-line images from **UTRNet's UTRSet-Real and UTRSet-Synth** datasets (ICDAR'23, Rahman, Ghosh, and Arora — IIIT Delhi) and folds a sample of them into your `labels.csv` file.

Where this fits with your five categories:
- **UTRSet-Real** → real scanned printed Urdu text lines from Rekhta Foundation book and document scans. This is a good fit for your **newspaper** and **book** categories.
- **UTRSet-Synth** → computer-generated Urdu text images. This is a good fit for your **synthetic** category.
- **Signboard** and **handwriting** are not covered by this dataset. There is no small, freely downloadable public dataset for Urdu scene-text (signboards) or handwriting that I could verify. For these two categories, your best options remain your own photos for signboards and, if you want a shortcut, reaching out to CLE Pakistan for their handwriting corpus.

**License note:** UTRSet is released under CC BY-NC-SA 4.0 for academic and research use. If you use it, cite the UTRNet paper from their GitHub README in your project report.

Run the cells in order. Step 3 matters: the exact internal folder layout is not something I could verify from here, so check the printed output before running Step 4.


## Step 1: Install gdown and download

`gdown` is the standard tool for downloading files from Google Drive from the command line.


In [1]:
!pip install --upgrade "transformers==4.57.6" torch pillow pandas gdown sentencepiece protobuf --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, urllib.request

os.makedirs("fonts", exist_ok=True)

FONT_URLS = {
    "fonts/NotoNastaliqUrdu.ttf":
        "https://raw.githubusercontent.com/google/fonts/main/ofl/notonastaliqurdu/NotoNastaliqUrdu%5Bwght%5D.ttf",
    "fonts/NotoNaskhArabic-Variable.ttf":
        "https://raw.githubusercontent.com/google/fonts/main/ofl/notonaskharabic/NotoNaskhArabic%5Bwght%5D.ttf",
}
for dest, url in FONT_URLS.items():
    if os.path.exists(dest):
        print(f"Already have {dest}, skipping")
        continue
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"Downloaded {dest} ({os.path.getsize(dest)/1024:.0f} KB)")
    except Exception as e:
        print(f"Could not download {dest}: {e}")

Downloaded fonts/NotoNastaliqUrdu.ttf (674 KB)
Downloaded fonts/NotoNaskhArabic-Variable.ttf (300 KB)


In [3]:
import tarfile, glob, urllib.error

os.makedirs("corpus", exist_ok=True)
url = "https://raw.githubusercontent.com/mirfan899/Urdu/master/news/headlines.csv.tar.gz"
archive_path = "corpus/headlines.csv.tar.gz"

try:
    urllib.request.urlretrieve(url, archive_path)
    with tarfile.open(archive_path) as tf:
        tf.extractall("corpus")
    matches = glob.glob("corpus/**/headlines.csv", recursive=True)
    if not matches:
        raise FileNotFoundError("headlines.csv not found inside the downloaded archive")
    headlines_csv = matches[0]
    print("Corpus ready:", headlines_csv)
except (urllib.error.URLError, tarfile.TarError, FileNotFoundError) as e:
    raise RuntimeError(
        f"Couldn't prepare the headlines corpus from {url}: {e}. "
        "Check your network connection before continuing to Step 2."
    ) from e

Corpus ready: corpus/headlines.csv


## Step 2: Generate Synthetic Urdu Text Images

Filters the headline corpus down to safe, single-line text, renders each line as a standalone image (mixing fonts and backgrounds), and appends the results to `labels.csv`.

In [4]:
import pandas as pd

SAFE_CATEGORIES = ["science", "health", "weird news", "sports"]  # skip politics/entertainment gossip

def load_safe_headlines(csv_path: str, n_needed: int, seed: int = 42) -> list[str]:
    """Sample a pool of short, single-line headlines from safe categories.

    Filters out politics/entertainment gossip, keeps line lengths in a
    range that renders well as a single text line (15-70 characters),
    drops rows containing embedded newlines/tabs, and de-duplicates
    before sampling.
    """
    df = pd.read_csv(csv_path, sep="\t")
    df = df[df["category"].isin(SAFE_CATEGORIES)]
    pool = df["title"].dropna().astype(str).str.strip()
    pool = pool[(pool.str.len() >= 15) & (pool.str.len() <= 70)]
    pool = pool[~pool.str.contains(r"[\t\n\r]")]
    pool = pool.drop_duplicates()
    return pool.sample(n=min(n_needed, len(pool)), random_state=seed).tolist()

In [5]:
from PIL import Image, ImageDraw, ImageFont
import random

FONTS = {"nastaliq": "fonts/NotoNastaliqUrdu.ttf", "naskh": "fonts/NotoNaskhArabic-Variable.ttf"}
BACKGROUNDS = [("white", (255, 255, 255)), ("cream", (250, 244, 227)), ("light_grey", (238, 238, 235))]

def render_line(text: str, font_path: str, font_size: int, bg_rgb: tuple[int, int, int], pad: int = 18) -> Image.Image:
    """Render a single line of RTL text as a standalone image.

    Uses Pillow's raqm-backed shaping (direction="rtl", language="ur") so
    Urdu letterforms join and reorder correctly, rather than being drawn
    as isolated, left-to-right glyphs.
    """
    font = ImageFont.truetype(font_path, font_size)
    tmp = Image.new("RGB", (10, 10))
    d = ImageDraw.Draw(tmp)
    bbox = d.textbbox((0, 0), text, font=font, direction="rtl", language="ur")
    w, h = (bbox[2] - bbox[0]) + 2 * pad, (bbox[3] - bbox[1]) + 2 * pad
    img = Image.new("RGB", (w, h), bg_rgb)
    draw = ImageDraw.Draw(img)
    draw.text((w - pad, pad - bbox[1]), text, font=font, fill=(20, 20, 20),
               direction="rtl", language="ur", anchor="ra")
    return img

In [ ]:
random.seed(42)

DATA_DIR = "/workspaces/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran/SI26-Week1/data"
LABELS_PATH = os.path.join(DATA_DIR, "labels.csv")
DEST_DIR = os.path.join(DATA_DIR, "raw", "synthetic")
os.makedirs(DEST_DIR, exist_ok=True)

N_SYNTH = 110  # adjust based on what the audit cell below says you still need

texts = load_safe_headlines(headlines_csv, N_SYNTH)
rows = []
for i, text in enumerate(texts):
    font_name = random.choice(list(FONTS.keys()))
    img = render_line(text, FONTS[font_name], random.randint(34, 56), random.choice(BACKGROUNDS)[1])
    fname = f"synth_{i:04d}.jpg"
    img.save(os.path.join(DEST_DIR, fname), quality=92)
    rows.append({"image": os.path.join("raw", "synthetic", fname), "text": text, "category": "synthetic"})

new_df = pd.DataFrame(rows)
print(f"Generated {len(new_df)} new images")
new_df.head()

In [ ]:
existing_df = pd.read_csv(LABELS_PATH)
combined_df = pd.concat([existing_df, new_df], ignore_index=True).drop_duplicates(subset=["image"])
combined_df.to_csv(LABELS_PATH, index=False)
print(f"labels.csv: {len(existing_df)} -> {len(combined_df)} rows")

labels.csv: 263 -> 263 rows


## Step 3: Inspect the folder structure (do this before Step 4)

Google Drive ZIP files do not always unpack into a flat, predictable layout. This cell prints the folder tree and looks for anything that could be a ground-truth label file (usually a `.txt` file mapping image paths to text). Read the output before touching Step 4.


In [ ]:
def check_zip_valid(path: str, min_size_mb: float = 1) -> bool:
    """Check whether a downloaded zip looks real (exists and isn't a tiny error page).

    Note: nothing earlier in this notebook actually calls gdown.download(...)
    for UTRSet-Real.zip / UTRSet-Synth.zip, even though gdown gets installed
    in Step 1 and this cell checks for the results. If you get Drive access
    to the real UTRSet files, add a gdown.download(...) call in Step 1 -
    right now this will always report them as missing.
    """
    if not os.path.exists(path) or os.path.getsize(path) / 1e6 < min_size_mb:
        print(f"Skipping {path} — download didn't come through (Drive quota or permissions). "
              f"That's fine, your synthetic images above already cover the 200+ target.")
        return False
    return True

utrset_real_ok = check_zip_valid("UTRSet-Real.zip")
utrset_synth_ok = check_zip_valid("UTRSet-Synth.zip")

Skipping UTRSet-Real.zip — download didn't come through (Drive quota or permissions). That's fine, your synthetic images above already cover the 200+ target.
Skipping UTRSet-Synth.zip — download didn't come through (Drive quota or permissions). That's fine, your synthetic images above already cover the 200+ target.


In [ ]:
df = pd.read_csv(LABELS_PATH)
print(f"Total rows: {len(df)}  (target: 200+, {'met' if len(df) >= 200 else f'need {200-len(df)} more'})")
print(df["category"].value_counts(), "\n")

missing = [row["image"] for _, row in df.iterrows() if not os.path.isfile(os.path.join(DATA_DIR, row["image"]))]
print(f"Missing files: {len(missing)}")
dupes = df.duplicated(subset=["image"]).sum()
print(f"Duplicate rows: {dupes}")

Total rows: 263  (target: 200+, met)
category
synthetic    110
Name: count, dtype: int64 

Missing files: 153
Duplicate rows: 0


## Step 4: Visualize the Dataset

Before this feeds into TrOCR, it's worth actually looking at what `labels.csv` and `DATA_DIR` contain rather than trusting the printed counts from Step 3 alone:

- **Category balance** — how many rows fall into each of the five categories, and whether any rows are missing a category value entirely.
- **File availability** — how many rows point to an image that doesn't actually exist yet (e.g. because the UTRSet zips didn't download).
- **Text length** — the distribution of line lengths, since very short or very long lines behave differently during OCR training.
- **A visual sample** — actual generated images next to their ground-truth text, as a sanity check that the labels match what's rendered.

This section only reads existing data — it doesn't regenerate or overwrite anything.

In [ ]:
import matplotlib.pyplot as plt

df["_available"] = df["image"].apply(lambda p: os.path.isfile(os.path.join(DATA_DIR, p)))

# dropna=False on purpose: a blank/missing category is a real data-quality
# issue and should show up as its own bar, not disappear silently
cat_counts = df["category"].value_counts(dropna=False)
cat_labels = ["(missing category)" if pd.isna(c) else c for c in cat_counts.index]

avail_counts = df["_available"].value_counts().reindex([True, False], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

bar_colors = ["#C44E52" if lbl == "(missing category)" else "#4C72B0" for lbl in cat_labels]
axes[0].bar(cat_labels, cat_counts.values, color=bar_colors)
axes[0].set_title("Samples per Category")
axes[0].set_ylabel("Number of Rows")
axes[0].tick_params(axis="x", rotation=25)
for i, v in enumerate(cat_counts.values):
    axes[0].text(i, v, str(v), ha="center", va="bottom", fontsize=9)

axes[1].bar(["Available", "Missing"], avail_counts.values, color=["#55A868", "#C44E52"])
axes[1].set_title("Image File Availability")
axes[1].set_ylabel("Number of Rows")
for i, v in enumerate(avail_counts.values):
    axes[1].text(i, v, str(v), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

n_missing_category = df["category"].isna().sum()
if n_missing_category:
    print(f"Note: {n_missing_category} row(s) in labels.csv have no category value — worth tracking down.")

In [ ]:
lengths = df["text"].astype(str).str.len()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(lengths, bins=15, color="#4C72B0", edgecolor="white")
ax.axvline(lengths.mean(), color="#C44E52", linestyle="--", label=f"Mean: {lengths.mean():.0f} characters")
ax.set_title("Text Length Distribution")
ax.set_xlabel("Characters per Line")
ax.set_ylabel("Number of Rows")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import textwrap

def show_sample_grid(data, data_dir, n=9, cols=3, font_path=FONTS["nastaliq"],
                      thumb_size=230, caption_h=92, font_size=16):
    """Display a grid of sample images with their ground-truth text as captions.

    Captions are drawn with PIL using the same RTL/shaping setup as the
    synthetic image generation above, then handed to matplotlib as a single
    raster image — matplotlib's own text renderer does not shape
    Arabic-script text correctly, so it can't draw the captions directly.
    """
    if len(data) == 0:
        print("No available images to sample from.")
        return

    sample = data.sample(n=min(n, len(data)), random_state=0)
    n_cols = min(cols, len(sample))
    n_rows = -(-len(sample) // n_cols)
    cell_w, cell_h = thumb_size, thumb_size + caption_h
    line_h = int(font_size * 2.0)  # Nastaliq needs generous line spacing (diagonal flow)

    grid_img = Image.new("RGB", (n_cols * cell_w, n_rows * cell_h), (255, 255, 255))
    font = ImageFont.truetype(font_path, font_size)

    for i, (_, row) in enumerate(sample.iterrows()):
        r, c = divmod(i, n_cols)
        thumb = Image.open(os.path.join(data_dir, row["image"])).convert("RGB")
        thumb.thumbnail((thumb_size - 10, thumb_size - 10))
        cell = Image.new("RGB", (cell_w, cell_h), (255, 255, 255))
        cell.paste(thumb, ((cell_w - thumb.width) // 2, (thumb_size - thumb.height) // 2))

        draw = ImageDraw.Draw(cell)
        wrapped = textwrap.wrap(str(row["text"]), width=24)
        lines = wrapped[:2]
        if len(wrapped) > 2:
            lines[-1] = lines[-1] + "…"
        for li, line in enumerate(lines):
            bbox = draw.textbbox((0, 0), line, font=font, direction="rtl", language="ur")
            tw = bbox[2] - bbox[0]
            y = thumb_size + 6 + li * line_h
            draw.text(((cell_w + tw) / 2, y), line, font=font, fill=(20, 20, 20),
                       direction="rtl", language="ur", anchor="ra")
        draw.rectangle([0, 0, cell_w - 1, cell_h - 1], outline=(220, 220, 220))
        grid_img.paste(cell, (c * cell_w, r * cell_h))

    plt.figure(figsize=(n_cols * 3, n_rows * 3.7))
    plt.imshow(grid_img)
    plt.axis("off")
    plt.title(f"Sample of {len(sample)} Dataset Images")
    plt.tight_layout()
    plt.show()

show_sample_grid(df[df["_available"]], DATA_DIR, n=9)

## Step 5: Build and Instantiate the PyTorch Dataset

Wraps `labels.csv` in a `torch.utils.data.Dataset` (skipping any row whose image file isn't on disk yet — e.g. if the UTRSet zips above didn't download), loads the TrOCR processor, and splits into train/test.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor

class UrduOCRDataset(Dataset):
    """PyTorch Dataset over labels.csv, skipping rows with no image on disk yet.

    Each item returns TrOCR-ready pixel_values and tokenized labels for one
    text-line image.
    """

    def __init__(self, csv_path: str, processor: TrOCRProcessor, data_dir: str, max_length: int = 128):
        data = pd.read_csv(csv_path)
        has_file = data["image"].apply(lambda p: os.path.isfile(os.path.join(data_dir, p)))
        n_missing = int((~has_file).sum())
        if n_missing:
            print(f"Skipping {n_missing} rows with no image on disk yet (e.g. UTRSet zips not downloaded/extracted)")
        self.data = data[has_file].reset_index(drop=True)
        self.processor = processor
        self.data_dir = data_dir
        self.max_length = max_length
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> dict:
        row = self.data.iloc[idx]
        image_path = os.path.join(self.data_dir, row["image"])
        try:
            image = Image.open(image_path).convert("RGB")
        except (FileNotFoundError, OSError) as e:
            raise FileNotFoundError(f"Row {idx}: can't open '{image_path}' ({e})")

        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(
            str(row["text"]), padding="max_length", max_length=self.max_length, truncation=True
        ).input_ids
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}

In [ ]:
import transformers
print(transformers.__version__)
assert transformers.__version__.startswith("4."), "Still on transformers 5.x — restart the kernel and re-run from the pip install cell"

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed", use_fast=True)
dataset = UrduOCRDataset(LABELS_PATH, processor, data_dir=DATA_DIR)

sample = dataset[0]
print("pixel_values:", sample["pixel_values"].shape, "| labels:", sample["labels"].shape)

train_size = int(0.8 * len(dataset))
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, len(dataset) - train_size])
print(f"Train: {train_size}  Test: {len(dataset) - train_size}")

4.57.6


Using `use_fast=True` but `torchvision` is not available. Falling back to the slow image processor.


Skipping 153 rows with no image on disk yet (e.g. UTRSet zips not downloaded/extracted)
Dataset loaded: 110 samples
pixel_values: torch.Size([3, 384, 384]) | labels: torch.Size([128])
Train: 88  Test: 22


## Step 6: Create DataLoaders and Preview a Batch

Wraps the train/test splits in `DataLoader`s and pulls one batch to confirm the tensor shapes look right before this feeds into a training loop.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

batch = next(iter(train_loader))
print("Batch pixel_values:", batch["pixel_values"].shape, "| Batch labels:", batch["labels"].shape)
print(f"Train batches: {len(train_loader)}  Test batches: {len(test_loader)}")

Batch pixel_values: torch.Size([8, 3, 384, 384]) | Batch labels: torch.Size([8, 128])
Train batches: 11  Test batches: 3


## Summary

This notebook expanded `labels.csv` with freshly generated synthetic Urdu text-line images, checked for the real UTRSet-Real/UTRSet-Synth downloads (Step 3), and visualized category balance, file availability, text length, and a sample of the images (Step 4). The `DataLoader`s built above are ready to feed into a TrOCR fine-tuning loop next.